Grid Search is one of the first steps that needs to be done. Grid Search involves searching through all possible weights that you could be used for training.

In [ ]:
import torch
import torch.optim as optim
import pandas as pd
import numpy as np
import sys
import os
import random
import time
import json
from datetime import timedelta, datetime
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, '/home/exouser/multi-objective-pdbbind/multi-objective-pdbbind')
sys.path.insert(0, '/home/exouser/multi-objective-pdbbind/multi-objective-pdbbind/pytorch-implementation/pdbbind')

from models.layers_pytorch_pdbbind import PGGCNModel
# Also removed hydrogen atoms from the dataset
from train_split_data import load_data_with_saved_split

We first set up Configurations of the grid search.

In [ ]:

# ============================================================================
# CONFIG — change SUBSET_SIZE to switch dataset size
# ============================================================================

class Config:
    # -------------------------------------------------------------------------
    # CHANGE THIS to switch dataset size: 100, 250, 500, 1000, 1500, 2000, 2660
    SUBSET_SIZE = 100
    # -------------------------------------------------------------------------

    SUBSET_DIR = '/home/exouser/multi-objective-pdbbind/multi-objective-pdbbind/Datasets/subsets'
    CSV_PATH   = '/home/exouser/multi-objective-pdbbind/multi-objective-pdbbind/Datasets/pdbbind.csv'
    PKL_PATH   = '/home/exouser/multi-objective-pdbbind/multi-objective-pdbbind/Datasets/PDBBind_full.pkl'

    # Model architecture — matches train_subset.py
    NUM_ATOM_FEATURES = 36
    R_OUT_CHANNEL     = 20
    C_OUT_CHANNEL     = 1024
    DROPOUT_RATE      = 0.05   # tuned from sweep

    # Training hyperparameters — tuned from sweep
    EPOCHS      = 300
    BATCH_SIZE  = 8
    LEARNING_RATE = 5e-4
    L2_WEIGHT   = 1e-4
    MAX_NORM    = 3.0

    # Physics weights to search — 101 values from 0.0 to 1.0
    PHYSICS_WEIGHTS = np.arange(0.0, 1.01, 0.01)

    RANDOM_SEED = 50

    @property
    def split_path(self):
        return os.path.join(self.SUBSET_DIR, f'pdbbind_subset_{self.SUBSET_SIZE}.pkl')

    @property
    def save_dir(self):
        return (
            '/home/exouser/multi-objective-pdbbind/multi-objective-pdbbind/'
            f'pytorch-implementation/pdbbind/grid_search/subset_{self.SUBSET_SIZE}'
        )

